In [0]:
# ============================================================
# NOTEBOOK: bronze_produtos_categorias
# Squad 1 · Dupla 3 — Bruna Diniz & Rafael Miranda
# ============================================================

In [0]:
%pip install deltalake --quiet
dbutils.library.restartPython()

In [0]:
from dotenv import load_dotenv
import os
from pyspark.sql import functions as F
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient
from deltalake import DeltaTable, write_deltalake

load_dotenv(".env")

TENANT_ID = os.getenv("ADLS_TENANT_ID")
CLIENT_ID = os.getenv("ADLS_CLIENT_ID")
CLIENT_SECRET = os.getenv("ADLS_CLIENT_SECRET")

STORAGE_ACCOUNT = "internshipdatalake"
CONTAINER = "raw"

adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": CLIENT_ID,
    "fs.azure.account.oauth2.client.secret": CLIENT_SECRET,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token",
}

base_url = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

credential = ClientSecretCredential(TENANT_ID, CLIENT_ID, CLIENT_SECRET)
datalake_service = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential=credential,
)
file_system_client = datalake_service.get_file_system_client(CONTAINER)

SQUAD_CONTAINER = "squad1"
lakehouse_base = f"abfss://{SQUAD_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

bronze_produtos_path = f"{lakehouse_base}/bronze/ecommerce_produtos"
bronze_categorias_path = f"{lakehouse_base}/bronze/ecommerce_categorias"

# Credenciais no formato que a deltalake (delta-rs) espera
storage_options = {
    "azure_storage_account_name": STORAGE_ACCOUNT,
    "azure_storage_client_id": CLIENT_ID,
    "azure_storage_client_secret": CLIENT_SECRET,
    "azure_storage_tenant_id": TENANT_ID,
}

print("Setup concluído.")

In [0]:
def listar_arquivos_pendentes(file_system_client, base_path, caminho_bronze, nome_arquivo):
    paths = file_system_client.get_paths(path=base_path, recursive=True)
    todos = [
        p.name for p in paths
        if not p.is_directory and p.name.endswith(f"{nome_arquivo}.parquet")
    ]
    try:
        dt = DeltaTable(caminho_bronze, storage_options=storage_options)
        df_existente = dt.to_pandas(columns=["bronze_source_file"])
        ja_processados = set(df_existente["bronze_source_file"].unique())
    except Exception:
        ja_processados = set()

    pendentes = [a for a in todos if a not in ja_processados]
    return pendentes



In [0]:
def ingerir_bronze(nome_arquivo, caminho_destino):
    pendentes = listar_arquivos_pendentes(file_system_client, "real-time-data", caminho_destino, nome_arquivo)
    print(f"[{nome_arquivo}] Arquivos novos encontrados: {len(pendentes)}")

    if not pendentes:
        print(f"[{nome_arquivo}] Nada novo para ingerir.")
        return

    for caminho_arquivo in pendentes:
        df_bruto = (
            spark.read.format("parquet").options(**adls_options)
            .load(f"{base_url}/{caminho_arquivo}")
            .withColumn("bronze_ingested_at", F.current_timestamp())
            .withColumn("bronze_source_file", F.lit(caminho_arquivo))
            .withColumn("_ano", F.date_format("bronze_ingested_at", "yyyy"))
            .withColumn("_mes", F.date_format("bronze_ingested_at", "MM"))
            .withColumn("_dia", F.date_format("bronze_ingested_at", "dd"))
            .withColumn("_hora", F.date_format("bronze_ingested_at", "HH"))
        )

        df_pandas = df_bruto.toPandas()

        # Corrige o timestamp pra ter timezone (UTC) -> evita exigir a
        # table feature "TimestampNtz" do delta-rs
        df_pandas["bronze_ingested_at"] = df_pandas["bronze_ingested_at"].dt.tz_localize("UTC")

        write_deltalake(
            caminho_destino,
            df_pandas,
            mode="append",
            partition_by=["_ano", "_mes", "_dia", "_hora"],
            storage_options=storage_options,
        )

        print(f"OK [{nome_arquivo}] {caminho_arquivo} - {len(df_pandas)} linhas")
  

In [0]:
squad_fs_client = datalake_service.get_file_system_client("squad1")

def limpar_caminho(caminho_relativo):
    try:
        dir_client = squad_fs_client.get_directory_client(caminho_relativo)
        dir_client.delete_directory()
        print(f"Apagado: {caminho_relativo}")
    except Exception as e:
        print(f"Nada pra apagar (ou erro): {e}")

limpar_caminho("bronze/ecommerce_produtos")
limpar_caminho("bronze/ecommerce_categorias")

In [0]:
ingerir_bronze("ecommerce_produtos", bronze_produtos_path)
ingerir_bronze("ecommerce_categorias", bronze_categorias_path)

In [0]:
dt_prod = DeltaTable(bronze_produtos_path, storage_options=storage_options)
dt_cat = DeltaTable(bronze_categorias_path, storage_options=storage_options)
print("Produtos:", len(dt_prod.to_pandas()))
print("Categorias:", len(dt_cat.to_pandas()))

In [0]:
ingerir_bronze("ecommerce_produtos", bronze_produtos_path)
ingerir_bronze("ecommerce_categorias", bronze_categorias_path)